In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_clean.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_features.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

df.head()

Dataset shape: (30000, 25)
Missing values: 0


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,TARGET
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [13]:
# Create a copy for feature engineering

df_fe = df.copy()

payment_status_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

print("Feature engineering dataset:", df_fe.shape)

Feature engineering dataset: (30000, 25)


In [14]:
# =========================================
# PAYMENT BEHAVIOR FEATURES
# =========================================

# Chỉ phần PAY > 0 được xem là số tháng chậm trả.
# PAY <= 0 được đưa về 0 cho các feature delinquency mới.
pay_delay = df_fe[payment_status_cols].clip(lower=0)

# Số tháng có xảy ra chậm trả trong 6 tháng
df_fe["DELAYED_MONTHS"] = (
    pay_delay > 0
).sum(axis=1)

# Mức chậm trả cao nhất từng ghi nhận
df_fe["MAX_DELAY"] = (
    pay_delay.max(axis=1)
)

# Mức chậm trả trung bình
df_fe["AVG_DELAY"] = (
    pay_delay.mean(axis=1)
)

# Có từng chậm trả hay chưa
df_fe["HAS_DELAY"] = (
    df_fe["DELAYED_MONTHS"] > 0
).astype(int)

# Trạng thái chậm trả gần nhất
df_fe["RECENT_DELAY"] = (
    df_fe["PAY_0"]
    .clip(lower=0)
)

df_fe[
    [
        "DELAYED_MONTHS",
        "MAX_DELAY",
        "AVG_DELAY",
        "HAS_DELAY",
        "RECENT_DELAY",
        "TARGET"
    ]
].head(10)

,DELAYED_MONTHS,MAX_DELAY,AVG_DELAY,HAS_DELAY,RECENT_DELAY,TARGET
0,2,2,0.666667,1,2,1
1,2,2,0.666667,1,0,1
2,0,0,0.000000,0,0,0
3,0,0,0.000000,0,0,0
4,0,0,0.000000,0,0,0
5,0,0,0.000000,0,0,0
6,0,0,0.000000,0,0,0
7,0,0,0.000000,0,0,0
8,1,2,0.333333,1,0,0
9,0,0,0.000000,0,0,0


In [15]:
# =========================================
# FINANCIAL BEHAVIOR FEATURES
# =========================================

# Trung bình số dư hóa đơn 6 tháng
df_fe["AVG_BILL_AMT"] = (
    df_fe[bill_cols].mean(axis=1)
)

# Trung bình số tiền thanh toán 6 tháng
df_fe["AVG_PAY_AMT"] = (
    df_fe[payment_cols].mean(axis=1)
)

# Tổng hóa đơn và tổng thanh toán
df_fe["TOTAL_BILL_AMT"] = (
    df_fe[bill_cols].sum(axis=1)
)

df_fe["TOTAL_PAY_AMT"] = (
    df_fe[payment_cols].sum(axis=1)
)

# Credit utilization proxy:
# Average Bill / Credit Limit
df_fe["CREDIT_UTILIZATION"] = (
    df_fe["AVG_BILL_AMT"]
    / df_fe["LIMIT_BAL"]
)

# Payment-to-Bill ratio
# Chỉ tính khi AVG_BILL_AMT > 0 để tránh chia 0 / bill âm
df_fe["PAYMENT_TO_BILL"] = np.where(
    df_fe["AVG_BILL_AMT"] > 0,
    df_fe["AVG_PAY_AMT"] / df_fe["AVG_BILL_AMT"],
    np.nan
)

df_fe[
    [
        "LIMIT_BAL",
        "AVG_BILL_AMT",
        "AVG_PAY_AMT",
        "TOTAL_BILL_AMT",
        "TOTAL_PAY_AMT",
        "CREDIT_UTILIZATION",
        "PAYMENT_TO_BILL"
    ]
].head(10)

,LIMIT_BAL,AVG_BILL_AMT,AVG_PAY_AMT,TOTAL_BILL_AMT,TOTAL_PAY_AMT,CREDIT_UTILIZATION,PAYMENT_TO_BILL
0,20000,1284.000000,114.833333,7704,689,0.064200,0.089434
1,120000,2846.166667,833.333333,17077,5000,0.023718,0.292791
2,90000,16942.166667,1836.333333,101653,11018,0.188246,0.108388
3,50000,38555.666667,1398.000000,231334,8388,0.771113,0.036259
4,50000,18223.166667,9841.500000,109339,59049,0.364463,0.540054
5,50000,39685.666667,1295.333333,238114,7772,0.793713,0.032640
6,500000,454099.166667,30126.500000,2724595,180759,0.908198,0.066343
7,100000,2247.666667,798.500000,13486,4791,0.022477,0.355257
8,140000,10868.666667,1126.833333,65212,6761,0.077633,0.103677
9,20000,4486.500000,2354.833333,26919,14129,0.224325,0.524871


In [16]:
# =========================================
# TREND FEATURES
# =========================================

# Bill gần đây so với 6 tháng trước
df_fe["BILL_TREND"] = (
    df_fe["BILL_AMT1"]
    - df_fe["BILL_AMT6"]
)

# Payment gần đây so với 6 tháng trước
df_fe["PAYMENT_TREND"] = (
    df_fe["PAY_AMT1"]
    - df_fe["PAY_AMT6"]
)

df_fe[
    [
        "BILL_AMT1",
        "BILL_AMT6",
        "BILL_TREND",
        "PAY_AMT1",
        "PAY_AMT6",
        "PAYMENT_TREND"
    ]
].head(10)

,BILL_AMT1,BILL_AMT6,BILL_TREND,PAY_AMT1,PAY_AMT6,PAYMENT_TREND
0,3913,0,3913,0,0,0
1,2682,3261,-579,0,2000,-2000
2,29239,15549,13690,1518,5000,-3482
3,46990,29547,17443,2000,1000,1000
4,8617,19131,-10514,2000,679,1321
5,64400,20024,44376,2500,800,1700
6,367965,473944,-105979,55000,13770,41230
7,11876,567,11309,380,1542,-1162
8,11285,3719,7566,3329,1000,2329
9,0,13912,-13912,0,0,0


In [17]:
new_features = [
    "DELAYED_MONTHS",
    "MAX_DELAY",
    "AVG_DELAY",
    "HAS_DELAY",
    "RECENT_DELAY",
    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "TOTAL_BILL_AMT",
    "TOTAL_PAY_AMT",
    "CREDIT_UTILIZATION",
    "PAYMENT_TO_BILL",
    "BILL_TREND",
    "PAYMENT_TREND"
]

print("Số feature mới:", len(new_features))
print("Số cột hiện tại:", df_fe.shape[1])

df_fe[new_features].describe().T.round(3)

Số feature mới: 13
Số cột hiện tại: 38


,count,mean,std,min,25%,50%,75%,max
DELAYED_MONTHS,30000.0,0.834,1.554,0.000,0.000,0.000,1.000,6.000
MAX_DELAY,30000.0,0.682,1.074,0.000,0.000,0.000,2.000,8.000
AVG_DELAY,30000.0,0.281,0.602,0.000,0.000,0.000,0.333,6.000
HAS_DELAY,30000.0,0.336,0.472,0.000,0.000,0.000,1.000,1.000
RECENT_DELAY,30000.0,0.357,0.761,0.000,0.000,0.000,0.000,8.000
AVG_BILL_AMT,30000.0,44976.945,63260.722,-56043.167,4781.333,21051.833,57104.417,877313.833
AVG_PAY_AMT,30000.0,5275.232,10137.946,0.000,1113.292,2397.167,5583.917,627344.333
TOTAL_BILL_AMT,30000.0,269861.671,379564.331,-336259.000,28688.000,126311.000,342626.500,5263883.000
TOTAL_PAY_AMT,30000.0,31651.393,60827.678,0.000,6679.750,14383.000,33503.500,3764066.000
CREDIT_UTILIZATION,30000.0,0.373,0.352,-0.233,0.030,0.285,0.688,5.364


In [18]:
print("NaN trong feature mới:")

print(
    df_fe[new_features]
    .isnull()
    .sum()
)

print("\nInfinity:")

print(
    np.isinf(
        df_fe[
            [
                "CREDIT_UTILIZATION",
                "PAYMENT_TO_BILL"
            ]
        ].select_dtypes(include=np.number)
    ).sum()
)

NaN trong feature mới:
DELAYED_MONTHS           0
MAX_DELAY                0
AVG_DELAY                0
HAS_DELAY                0
RECENT_DELAY             0
AVG_BILL_AMT             0
AVG_PAY_AMT              0
TOTAL_BILL_AMT           0
TOTAL_PAY_AMT            0
CREDIT_UTILIZATION       0
PAYMENT_TO_BILL       1071
BILL_TREND               0
PAYMENT_TREND            0
dtype: int64

Infinity:
CREDIT_UTILIZATION    0
PAYMENT_TO_BILL       0
dtype: int64


In [19]:
# =========================================
# MODEL FEATURE SELECTION
# =========================================

exclude_from_model = [
    "ID",
    "TARGET",
    "SEX",
    "AGE",
    "MARRIAGE"
]

model_features = [
    col
    for col in df_fe.columns
    if col not in exclude_from_model
]

print("Total columns:", df_fe.shape[1])
print("Model features:", len(model_features))

print("\nExcluded from model:")
for col in exclude_from_model:
    print("-", col)

print("\nFeatures used for modeling:")
for i, col in enumerate(model_features, start=1):
    print(f"{i:02d}. {col}")

Total columns: 38
Model features: 33

Excluded from model:
- ID
- TARGET
- SEX
- AGE
- MARRIAGE

Features used for modeling:
01. LIMIT_BAL
02. EDUCATION
03. PAY_0
04. PAY_2
05. PAY_3
06. PAY_4
07. PAY_5
08. PAY_6
09. BILL_AMT1
10. BILL_AMT2
11. BILL_AMT3
12. BILL_AMT4
13. BILL_AMT5
14. BILL_AMT6
15. PAY_AMT1
16. PAY_AMT2
17. PAY_AMT3
18. PAY_AMT4
19. PAY_AMT5
20. PAY_AMT6
21. DELAYED_MONTHS
22. MAX_DELAY
23. AVG_DELAY
24. HAS_DELAY
25. RECENT_DELAY
26. AVG_BILL_AMT
27. AVG_PAY_AMT
28. TOTAL_BILL_AMT
29. TOTAL_PAY_AMT
30. CREDIT_UTILIZATION
31. PAYMENT_TO_BILL
32. BILL_TREND
33. PAYMENT_TREND


In [20]:
# =========================================
# FINAL FEATURE VALIDATION
# =========================================

print("Rows:", df_fe.shape[0])
print("Columns:", df_fe.shape[1])

print("\nTARGET distribution:")
print(df_fe["TARGET"].value_counts())

print("\nDuplicate IDs:")
print(df_fe["ID"].duplicated().sum())

print("\nMissing values in entire dataset:")
print(df_fe.isnull().sum()[df_fe.isnull().sum() > 0])

Rows: 30000
Columns: 38

TARGET distribution:
TARGET
0    23364
1     6636
Name: count, dtype: int64

Duplicate IDs:
0

Missing values in entire dataset:
PAYMENT_TO_BILL    1071
dtype: int64


In [21]:
# =========================================
# SAVE FEATURE DATASET
# =========================================

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_features.csv"
)

df_fe.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Feature dataset saved:")
print(OUTPUT_PATH)

print("\nFinal shape:", df_fe.shape)

Feature dataset saved:
h:\Credit_Risk_Modeling\data\processed\credit_default_features.csv

Final shape: (30000, 38)


## Feature Engineering Summary

New behavioral and financial features were created from the original credit history data.

### Payment behavior features
- DELAYED_MONTHS
- MAX_DELAY
- AVG_DELAY
- HAS_DELAY
- RECENT_DELAY

### Financial behavior features
- AVG_BILL_AMT
- AVG_PAY_AMT
- TOTAL_BILL_AMT
- TOTAL_PAY_AMT
- CREDIT_UTILIZATION
- PAYMENT_TO_BILL

### Trend features
- BILL_TREND
- PAYMENT_TREND

Customer ID is retained for identification but will not be used as a predictive feature.

SEX, AGE and MARRIAGE are retained for analysis and fairness monitoring but excluded from the primary credit risk model.

Missing PAYMENT_TO_BILL values will be handled inside the machine-learning preprocessing pipeline to avoid data leakage.